# Bài tập 2 - Hồng Gấm

Môn 121036. Người phụ trách: **Hồng Gấm**.

Hoàn thành nội bộ: **18/11/2026**. Hạn nộp: **20/11/2026**.

Bộ khung chưa có thuật toán hoặc kết quả thực nghiệm. Đọc `README.md`, đề Bài tập 2 và bảng phân công của nhóm.

## 0. Chuẩn bị môi trường

Cell bên dưới do Codex tạo để import thư viện và xác định đường dẫn, ngoài phần cài đặt thuật toán. Các cell cài đặt được để trống.

**Khai báo AI cho cell chuẩn bị**

- Công cụ: Codex (OpenAI).
- Mục đích: tạo bộ khung thư mục, notebook và đường dẫn dùng chung.
- Prompt nguyên văn:

> giờ bạn tạo cái sườn giúp tôi [https://github.com/NguyenTanHiep07/xla-121036](https://github.com/NguyenTanHiep07/xla-121036) làm tiếp trong này cho dễ nha tạo mấy thư mục mấy cũ ko liên quan á bỏ qua

In [73]:
from pathlib import Path
import math
import time

import numpy as np
import matplotlib.pyplot as plt

# Chuẩn bị đường dẫn, không cài đặt thuật toán.
thu_muc_chay = Path.cwd().resolve()
thu_muc_tim = (
    thu_muc_chay,
    thu_muc_chay / "bai_tap_2",
    thu_muc_chay.parent,
)
BT2_DIR = next(
    (p for p in thu_muc_tim if (p / "bt2_nhom.ipynb").is_file()),
    None,
)
if BT2_DIR is None:
    raise FileNotFoundError(
        "Chọn thư mục chạy là gốc repo, bai_tap_2 hoặc bai_tap_2/lam_viec."
    )
IMAGES_DIR = BT2_DIR.parent / "images"
print("Bài tập 2:", BT2_DIR)
print("Ảnh dùng chung:", IMAGES_DIR)


Bài tập 2: C:\Users\ad\OneDrive\XLA\xla-121036-main\xla-121036-main\bai_tap_2
Ảnh dùng chung: C:\Users\ad\OneDrive\XLA\xla-121036-main\xla-121036-main\images


## Cài đặt - KHÔNG dùng AI

Tự viết các hàm ở những cell trống bên dưới. Chữ ký và đầu ra trong markdown là bản nhắc đặc tả của đề; không có mã thuật toán được tạo sẵn. Không dùng AI để viết, sửa lỗi hay viết lại phần này theo quy định của đề.

### `gaussian_blur(img, sigma)`

Làm mờ Gauss; được phép dùng scipy.ndimage.gaussian_filter.

In [74]:

def gaussian_blur(img, sigma):
    img = np.asarray(img, dtype=np.float64)

    if sigma <= 0:
        return img.copy()

    radius = int(np.ceil(3 * sigma))
    x = np.arange(-radius, radius + 1, dtype=np.float64)

    kernel_1d = np.exp(-(x ** 2) / (2 * sigma ** 2))
    kernel_1d /= kernel_1d.sum()

    kernel = np.outer(kernel_1d, kernel_1d)

    padded = np.pad(img, radius, mode="reflect")
    result = np.zeros_like(img)

    for i in range(kernel.shape[0]):
        for j in range(kernel.shape[1]):
            result += kernel[i, j] * padded[
                i:i + img.shape[0],
                j:j + img.shape[1]
            ]

    return result


### `structure_tensor(img, sigma_i, sigma_d)`

Trả (Mxx, Mxy, Myy), mỗi mảng cùng kích thước ảnh; có hệ số sigma_d**2 theo đề.

In [75]:

def structure_tensor(img, sigma_i, sigma_d):
    img = np.asarray(img, dtype=np.float64)

    blurred = gaussian_blur(img, sigma_d)

    Iy, Ix = np.gradient(blurred)

    factor = sigma_d ** 2

    Mxx = factor * gaussian_blur(Ix ** 2, sigma_i)
    Mxy = factor * gaussian_blur(Ix * Iy, sigma_i)
    Myy = factor * gaussian_blur(Iy ** 2, sigma_i)

    return Mxx, Mxy, Myy


### `harris_response(img, sigma_i, sigma_d, k=0.04)`

Trả đáp ứng R dạng mảng float cùng kích thước ảnh.

In [76]:

def harris_response(img, sigma_i, sigma_d, k=0.04):
    Mxx, Mxy, Myy = structure_tensor(
        img, sigma_i, sigma_d
    )

    det = Mxx * Myy - Mxy ** 2
    trace = Mxx + Myy

    R = det - k * trace ** 2

    return R


### `nms_2d(response, window=3)`

Trả mảng bool đánh dấu cực đại địa phương.

In [77]:

def nms_2d(response, window=3):
    response = np.asarray(response, dtype=np.float64)

    if window < 1 or window % 2 == 0:
        raise ValueError("window phải là số nguyên dương lẻ")

    radius = window // 2

    padded = np.pad(
        response,
        radius,
        mode="constant",
        constant_values=-np.inf
    )

    local_max = np.full_like(response, -np.inf)

    for i in range(window):
        for j in range(window):
            region = padded[
                i:i + response.shape[0],
                j:j + response.shape[1]
            ]
            local_max = np.maximum(local_max, region)

    return (response == local_max) & np.isfinite(response)


### `harris_classic(img, sigma_i=2.0, sigma_d=1.4, k=0.04, thresh_rel=0.01)`

Trả mảng (N, 3): (row, col, response), giảm dần theo response.

In [78]:

def harris_classic(
    img,
    sigma_i=2.0,
    sigma_d=1.4,
    k=0.04,
    thresh_rel=0.01
):
    R = harris_response(img, sigma_i, sigma_d, k)

    max_response = np.max(R)

    if max_response <= 0:
        return np.empty((0, 3), dtype=np.float64)

    mask = nms_2d(R, window=3)
    mask &= R > thresh_rel * max_response
    mask &= R > 0

    rows, cols = np.where(mask)
    responses = R[rows, cols]

    points = np.column_stack((rows, cols, responses))

    order = np.argsort(points[:, 2])[::-1]

    return points[order]


### `repeatability(pts_a, pts_b, transform, img_shape, eps=3.0)`

Tỉ lệ trong [0, 1]. transform ánh xạ (row, col) của A sang B; chỉ tính điểm chiếu còn trong ảnh B.

In [79]:

def repeatability(pts_a, pts_b, transform, img_shape, eps=3.0):
    pts_a = np.asarray(pts_a, dtype=np.float64)
    pts_b = np.asarray(pts_b, dtype=np.float64)

    if pts_a.size == 0 or pts_b.size == 0:
        return 0.0

    coords_a = pts_a.reshape(-1, pts_a.shape[-1])[:, :2]
    coords_b = pts_b.reshape(-1, pts_b.shape[-1])[:, :2]

    if callable(transform):
        projected = np.asarray(
            transform(coords_a),
            dtype=np.float64
        ).reshape(-1, 2)
    else:
        H = np.asarray(transform, dtype=np.float64)

        if H.shape != (3, 3):
            raise ValueError("transform phải là ma trận 3x3")

        xy1 = np.column_stack((
            coords_a[:, 1],
            coords_a[:, 0],
            np.ones(len(coords_a))
        ))

        mapped = (H @ xy1.T).T

        projected = np.full(
            (len(coords_a), 2),
            np.nan,
            dtype=np.float64
        )

        valid_w = (
            np.isfinite(mapped[:, 2]) &
            (np.abs(mapped[:, 2]) > 1e-12)
        )

        projected[valid_w, 0] = (
            mapped[valid_w, 1] / mapped[valid_w, 2]
        )

        projected[valid_w, 1] = (
            mapped[valid_w, 0] / mapped[valid_w, 2]
        )

    h, w = img_shape[:2]

    valid = (
        np.isfinite(projected).all(axis=1) &
        (projected[:, 0] >= 0) &
        (projected[:, 0] < h) &
        (projected[:, 1] >= 0) &
        (projected[:, 1] < w)
    )

    projected = projected[valid]

    coords_b = coords_b[
        np.isfinite(coords_b).all(axis=1)
    ]

    if len(projected) == 0 or len(coords_b) == 0:
        return 0.0

    distances = np.linalg.norm(
        projected[:, None, :] -
        coords_b[None, :, :],
        axis=2
    )

    matches = np.min(distances, axis=1) <= eps

    return float(np.mean(matches))


In [80]:

# tu kiem tra 6 ham harris

img = np.zeros((64, 64), dtype=np.float64)
img[16:48, 16:48] = 255


# gaussian_blur

sigma = 1.4
blur = gaussian_blur(img, sigma)

print("\ngaussian_blur")
print("sigma:", sigma)
print("kich thuoc anh goc:", img.shape)
print("kich thuoc anh sau lam mo:", blur.shape)
print("gia tri min:", round(float(blur.min()), 4))
print("gia tri max:", round(float(blur.max()), 4))
print("gia tri trung binh:", round(float(blur.mean()), 4))
print("tong gia tri pixel:", round(float(blur.sum()), 4))


# structure_tensor

sigma_i = 2.0
sigma_d = 1.4

Mxx, Mxy, Myy = structure_tensor(img, sigma_i, sigma_d)

print("\nstructure_tensor")
print("sigma_i:", sigma_i)
print("sigma_d:", sigma_d)
print("he so sigma_d^2:", sigma_d ** 2)
print("kich thuoc Mxx:", Mxx.shape)
print("kich thuoc Mxy:", Mxy.shape)
print("kich thuoc Myy:", Myy.shape)
print("Mxx min:", round(float(Mxx.min()), 4))
print("Mxx max:", round(float(Mxx.max()), 4))
print("Mxy min:", round(float(Mxy.min()), 4))
print("Mxy max:", round(float(Mxy.max()), 4))
print("Myy min:", round(float(Myy.min()), 4))
print("Myy max:", round(float(Myy.max()), 4))


# harris_response

k = 0.04
R = harris_response(img, sigma_i, sigma_d, k)

print("\nharris_response")
print("k:", k)
print("kich thuoc response:", R.shape)
print("response min:", round(float(R.min()), 4))
print("response max:", round(float(R.max()), 4))
print("response trung binh:", round(float(R.mean()), 4))
print("so pixel response duong:", int(np.sum(R > 0)))
print("so pixel response am:", int(np.sum(R < 0)))


# nms_2d

window = 3

test_R = np.array([
    [0, 0, 0],
    [0, 5, 0],
    [0, 0, 0]
], dtype=np.float64)

mask = nms_2d(test_R, window)

print("\nnms_2d")
print("window:", window)
print("kich thuoc response:", test_R.shape)
print("kich thuoc mask:", mask.shape)
print("kieu du lieu mask:", mask.dtype)
print("so cuc dai cuc bo:", int(mask.sum()))
print("toa do cuc dai:", np.argwhere(mask).tolist())


# harris_classic

pts = harris_classic(
    img,
    sigma_i=2.0,
    sigma_d=1.4,
    k=0.04,
    thresh_rel=0.01
)

print("\nharris_classic")
print("sigma_i:", 2.0)
print("sigma_d:", 1.4)
print("k:", 0.04)
print("thresh_rel:", 0.01)
print("so diem goc phat hien:", len(pts))
print("kich thuoc mang diem:", pts.shape)
print("dinh dang: (row, col, response)")
print("5 diem co response cao nhat:")
print(np.round(pts[:5], 4))


# repeatability

pts_a = np.array([
    [10, 10, 1],
    [20, 20, 1],
    [30, 30, 1]
], dtype=np.float64)

pts_b = np.array([
    [10, 10, 1],
    [20, 20, 1],
    [40, 40, 1]
], dtype=np.float64)

H = np.eye(3)
eps = 3.0

score = repeatability(
    pts_a,
    pts_b,
    H,
    (64, 64),
    eps=eps
)

pts_outside = np.array([
    [10, 10, 1],
    [20, 20, 1],
    [100, 100, 1]
], dtype=np.float64)

score_outside = repeatability(
    pts_outside,
    pts_b,
    H,
    (64, 64),
    eps=eps
)

score_empty = repeatability(
    np.empty((0, 3)),
    pts_b,
    H,
    (64, 64),
    eps=eps
)

print("\nrepeatability")
print("eps:", eps)
print("kich thuoc anh:", (64, 64))
print("ma tran bien doi:")
print(H)
print("so diem anh a:", len(pts_a))
print("so diem anh b:", len(pts_b))
print("repeatability:", round(float(score), 4))
print("repeatability khi co diem ngoai anh:",
      round(float(score_outside), 4))
print("repeatability khi khong co diem:",
      round(float(score_empty), 4))


gaussian_blur
sigma: 1.4
kich thuoc anh goc: (64, 64)
kich thuoc anh sau lam mo: (64, 64)
gia tri min: 0.0
gia tri max: 255.0
gia tri trung binh: 63.75
tong gia tri pixel: 261120.0

structure_tensor
sigma_i: 2.0
sigma_d: 1.4
he so sigma_d^2: 1.9599999999999997
kich thuoc Mxx: (64, 64)
kich thuoc Mxy: (64, 64)
kich thuoc Myy: (64, 64)
Mxx min: 0.0
Mxx max: 4161.0689
Mxy min: -913.2408
Mxy max: 913.2408
Myy min: 0.0
Myy max: 4161.0689

harris_response
k: 0.04
kich thuoc response: (64, 64)
response min: -692579.7778
response max: 3099951.4905
response trung binh: -7789.4826
so pixel response duong: 572
so pixel response am: 2496

nms_2d
window: 3
kich thuoc response: (3, 3)
kich thuoc mask: (3, 3)
kieu du lieu mask: bool
so cuc dai cuc bo: 1
toa do cuc dai: [[1, 1]]

harris_classic
sigma_i: 2.0
sigma_d: 1.4
k: 0.04
thresh_rel: 0.01
so diem goc phat hien: 4
kich thuoc mang diem: (4, 3)
dinh dang: (row, col, response)
5 diem co response cao nhat:
[[4.60000000e+01 4.60000000e+01 3.09995149e

## Kiểm chứng cài đặt

Mã A1-A5 và B1-B5 bên dưới là thứ tự kiểm chứng trong mục 3.7 / 4.5 của đề. Điền dữ liệu kiểm tra, số đo và output thực tế. Các ô mã hiện để trống.

### Kiểm chứng A1 - Bàn cờ tổng hợp

Giao điểm trong, sai số định vị ≤ 1 px. Người phụ trách: Gấm.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét kiểm chứng:**

- Kết quả thực tế: 
- Đạt/chưa đạt và lý do: 

### Kiểm chứng A2 - Xoay ảnh 30 độ

Báo số điểm và repeatability bằng phép biến đổi tọa độ đúng. Người phụ trách: Gấm.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét kiểm chứng:**

- Kết quả thực tế: 
- Đạt/chưa đạt và lý do: 

### Kiểm chứng A4 - Đối chiếu OpenCV

Dùng cv2.cornerHarris trong cell đối chiếu; báo điểm khớp trong 2-3 px. Người phụ trách: Gấm.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét kiểm chứng:**

- Kết quả thực tế: 
- Đạt/chưa đạt và lý do: 

## Thí nghiệm - Rà số đo và điều kiện so sánh của ba cặp

Tự chọn dữ liệu, trục quét, số đo và kết luận. Theo mục 5.1, mỗi cặp cần bằng chứng khác biệt, quét tham số có ý nghĩa và giải thích cơ chế. Nếu thực tế dùng AI hỗ trợ cell thí nghiệm, ghi prompt nguyên văn ngay phía trên cell đó.

### Thiết kế và bằng chứng khác biệt

Nêu ảnh nào, vì sao chọn, giữ cố định điều kiện gì và đo đại lượng nào. Trình bày kết quả của hai phương pháp.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét từ số liệu đã chạy:**



### Quét tham số

Ghi trục quét, giá trị thử và các điều kiện giữ cố định; trình bày đường cong hoặc điểm chuyển hành vi.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét từ số liệu đã chạy:**



### Giải thích kết quả

Điền kết quả đo, giải thích đại lượng nào thay đổi, ở bước nào và vì sao. Báo cả trường hợp chưa cải thiện.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét từ số liệu đã chạy:**



## Bàn giao cho nhóm trưởng

- [ ] Đúng giao diện hàm và quy ước tọa độ.
- [ ] Restart & Run All chạy được, đã lưu output.
- [ ] Ảnh cần dùng có trong images/ tại gốc repo; dùng IMAGES_DIR để đọc ảnh.
- [ ] Kiểm chứng và thí nghiệm có số đo thực tế.
- [ ] Cell thí nghiệm có dùng AI ghi đúng prompt ngay phía trên.
- [ ] Đã mở Pull Request và ghi nguồn các hàm dùng chung.